In [1]:
import numpy as np
import os
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import Ridge
import torchaudio
import numpy as np
import pandas as pd
import sys
import os

import numpy as np
import os
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

import torch
import torchaudio

# Add parent directory to Python path
sys.path.append(os.path.abspath('..'))

import feature_extractor as fe
import feature_selection as fs

In [9]:
feature_extractor = fe.FeatureExtractor('hubert_l')
def covariance_lasso_selector(X, n_features_target=103):
    """
    X: Matriz de (muestras, 1024)
    n_features_target: Cuántas dimensiones queremos (103)
    """
    print("Escalando datos y calculando matriz de covarianza implícita...")
    
    # 1. Estandarizar es vital para que la covarianza sea justa
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # 2. Ajuste fino de Alpha para obtener exactamente n_features_target
    # Empezamos con un alpha pequeño y lo subimos hasta que queden 103
    alpha = 0.01
    found = False
    max_iter = 20
    
    print("Buscando el nivel de regularización óptimo...")
    for i in range(max_iter):
        # Lasso utiliza la estructura de covarianza para penalizar rasgos redundantes
        model = Lasso(alpha=alpha, max_iter=2000)
        model.fit(X_scaled, X_scaled) # Auto-selección predictiva
        
        # Contamos cuántos coeficientes NO son cero
        # Sumamos la importancia a través de todas las "salidas"
        importance = np.sum(np.abs(model.coef_), axis=0)
        selected_indices = np.where(importance > 0)[0]
        
        current_count = len(selected_indices)
        
        if current_count <= n_features_target:
            print(f"¡Logrado! Alpha {alpha:.4f} produjo {current_count} rasgos.")
            found = True
            break
        
        alpha *= 1.5 # Incrementamos la presión de la regularización
        
    # 3. Si Lasso fue muy agresivo o no lo suficiente, forzamos los mejores
    if len(selected_indices) > n_features_target:
        # Tomamos los 103 con mayores coeficientes (más importantes)
        top_indices = np.argsort(importance)[-n_features_target:]
        return np.sort(top_indices)
    
    return selected_indices

torch version: 2.9.1+cpu
torch audio version: 2.9.1+cpu
device: cpu
Sample Rate: 16000
model class: <class 'torchaudio.pipelines._wav2vec2.utils._Wav2Vec2Model'>


In [14]:
def run_covariance_pipeline(input_folder, output_folder):
    # Usamos el FeatureExtractor de 24 capas definido anteriormente
    # extractor = FeatureExtractor() 
    
    # 1. Cargar datos
    print("Extrayendo características de 24 capas...")
    # X_matrix = np.array([...]) # Matriz de (N, 1024)
    # file_names = [...]
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Listar archivos .wav que empiecen con ES_
    audio_files = [f for f in os.listdir(input_folder) if f.endswith('.wav') and f.startswith('ES_')]
    
    print(f"Encontrados {len(audio_files)} archivos para procesar.")

    all_features_1024 = []
    filenames_order = []

    print(f"Paso 1: Extrayendo características originales de {len(audio_files)} archivos...")

    for filename in audio_files:
        path = os.path.join(input_folder, filename)
        # Extraer vector de 1024
        feat_1024 = feature_extractor.get_24th_layer_features_averages(path, extract_with_padding=True)
        
        all_features_1024.append(feat_1024)
        filenames_order.append(filename)

    # Convertir a matriz de numpy (Número de audios, 1024)
    data_matrix = np.array(all_features_1024)
    # 2. Seleccionar los 103 ganadores
    indices_ganadores = covariance_lasso_selector(data_matrix, n_features_target=103)
    
    print(f"Dimensiones finales seleccionadas: {indices_ganadores}")
    
    # 3. Guardar con formato específico para el reto
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    for i, name in enumerate(filenames_order):
        # Aplicamos la máscara de los 103 seleccionados
        vector_103 = data_matrix[i, indices_ganadores]
        
        # Guardar como .npy con precisión double (<f8)
        save_name = name.replace("ES_", "EN_").replace(".wav", "_features.npy")
        np.save(os.path.join(output_folder, save_name), vector_103.astype(np.float64))

    print(f"Proceso finalizado. {len(filenames_order)} archivos generados.")

In [ ]:
# ==========================================
# 1. FUNCIÓN PARA CARGAR TUS DATOS (1024)
# ==========================================
# Asumimos que ya tienes una carpeta con archivos .npy de 1024 rasgos
# o que los extraes usando el FeatureExtractor anterior.
feature_extractor = fe.FeatureExtractor('wavlm_300')
english_winner = [1 , 8, 10, 26, 59, 66, 67, 87, 90, 97, 99,107,122,123,
                  126,128,137,144,151,155,157,161,169,178,180,184,194,200,
                  206,219,236,238,240,241,270,310,313,314,322,346,347,351,
                  357,363,367,377,380,390,399,403,410,420,425,426,429,431,
                  438,464,471,499,504,529,538,542,543,547,557,562,576,585,591,
                  604,616,618,635,648,650,667,668,687,704,735,738,739,752,767,
                  801,822,839,847,856,858,860,880,900,903,909,924,942,944,976,982,1022]
def greedy_selection_103(X_train, n_features=103):
    """
    X_train: Matriz de (n_muestras, 1024)
    Aplica Forward Sequential Feature Selection (Algoritmo Greedy)
    """
    print(f"Iniciando selección Greedy para encontrar los {n_features} mejores rasgos...")
    
    # Usamos Ridge como modelo base para evaluar la importancia
    # Ridge es rápido y maneja bien la colinealidad del audio
    estimator = Ridge(alpha=1.0)
    
    # SFS es "Greedy": añade el mejor rasgo, luego el siguiente mejor, etc.
    sfs = SequentialFeatureSelector(
        estimator, 
        n_features_to_select=n_features, 
        direction='forward', 
        scoring='neg_mean_squared_error',
        cv=3 # Cross-validation para asegurar robustez
    )
    
    sfs.fit(X_train, X_train) # Aquí intentamos reconstruir la señal consigo misma de forma eficiente
    
    indices = sfs.get_support(indices=True)
    return indices

# ==========================================
# 2. PROCESAMIENTO COMPLETO
# ==========================================
def run_greedy_pipeline(input_folder, output_folder):
    # 1. Extraer o cargar los vectores de 1024
    # (Usando la lógica de FeatureExtractor de los mensajes anteriores)
    # Supongamos que ya tenemos 'X_matrix' de forma (n_files, 1024)
    
    # Por ahora simulamos la matriz X_matrix con tus datos reales
    # X_matrix = np.array(lista_de_vectores_1024)
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Listar archivos .wav que empiecen con ES_
    audio_files = [f for f in os.listdir(input_folder) if f.endswith('.wav') and f.startswith('ES_')]
    
    print(f"Encontrados {len(audio_files)} archivos para procesar.")

    all_features_1024 = []
    filenames_order = []

    print(f"Paso 1: Extrayendo características originales de {len(audio_files)} archivos...")

    for filename in audio_files:
        path = os.path.join(input_folder, filename)
        # Extraer vector de 1024
        feat_1024 = feature_extractor.get_24th_layer_features_averages(path, extract_with_padding=True)
        
        all_features_1024.append(feat_1024)
        filenames_order.append(filename)

    # Convertir a matriz de numpy (Número de audios, 1024)
    data_matrix = np.array(all_features_1024)
    # 2. Obtener los 103 índices ganadores
    indices_ganadores =  greedy_selection_103(data_matrix) #english_winner 
    print(f"Índices seleccionados: {indices_ganadores}")

    # 3. Guardar solo esos rasgos
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for i, file_name in enumerate(filenames_order):
        # Seleccionamos solo las 103 columnas elegidas por el algoritmo greedy
        vector_103 =  data_matrix[i, indices_ganadores]
        
        # Guardar en formato .npy con precisión <f8
        save_path = os.path.join(output_folder, file_name.replace("ES_", "EN_").replace(".wav", "_features.npy"))
        np.save(save_path, vector_103.astype(np.float64))

    print("¡Proceso Greedy completado!")

In [15]:
data_dir = "../data"
pred_dir = os.path.join(data_dir, "spanish_test")
feature_dir = feature_directory = os.path.join(data_dir, "english")

In [ ]:
run_greedy_pipeline(pred_dir, feature_dir)

In [ ]:
run_covariance_pipeline(pred_dir, feature_dir)

Extrayendo características de 24 capas...
Encontrados 200 archivos para procesar.
Paso 1: Extrayendo características originales de 200 archivos...
